In [ ]:
from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import *
import os

In [ ]:
spark = (SparkSession
        .builder
        .appName("Spark Data Sources Example")
        .config("spark.jars.packages", "org.apache.spark:spark-avro_2.13:4.2.0") 
        .getOrCreate())

# Parquet

## Reading Parquet files into a DataFrame

In [ ]:
file = """flights/data/summary-data/parquet/2010-summary.parquet/"""
df = spark.read.format("parquet").load(file)


## Reading Parquet files into a Spark SQL Table

In [ ]:
spark.sql("""CREATE OR REPLACE TEMPORARY VIEW us_delay_flights_tbl
USING parquet
OPTIONS (
path "flights/data/summary-data/parquet/2010-summary.parquet/" )""")



In [ ]:
spark.sql("SELECT * FROM us_delay_flights_tbl").show()

## Writing DataFrames to Parquet files

In [ ]:
(df.write.format("parquet")
.mode("overwrite")
.option("compression", "snappy")
.save("tmp/parquet/df_parquet"))


## Writing DataFrames to Spark SQL tables
Writing a DataFrame to a SQL table is as easy as writing to a file—just use saveAsTable() instead of save(). This will create a managed table called
us_delay_flights_tbl:

In [ ]:
(df.write
.mode("overwrite")
.saveAsTable("test_us_delay_flights_tbl"))


# JSON

JavaScript Object Notation (JSON) is also a popular data format. It came to prominence as an easy-to-read and easy-to-parse format compared to XML. It has two representational formats: single-line mode and multiline mode. Both modes are
supported in Spark.


In [ ]:
file = "flights/data/summary-data/json/"
df = spark.read.format("json").load(file)


In [ ]:
spark.sql("""CREATE OR REPLACE TEMPORARY VIEW us_delay_flights_tbl
USING json
OPTIONS (
path "flights/data/summary-data/json/" )""")

In [ ]:
spark.sql("SELECT * FROM us_delay_flights_tbl").show()

In [ ]:
(df.write.format("json")
.mode("overwrite")
.option("compression", "snappy")
.save("tmp/json/df_json"))


## JSON data Source Options



| Property name | Values | Meaning | Scope |
| :--- | :--- | :--- | :--- |
| `compression` | none, uncompressed, bzip2, deflate, gzip, lz4, or snappy | Use this compression codec for writing. Note that read will only detect the compression or codec from the file extension. | Write |
| `dateFormat` | yyyy-MM-dd or DateTimeFormatter | Use this format or any format from Java's DateTimeFormatter. | Read/write |
| `multiLine` | true, false | Use multiline mode. Default is false (single-line mode). | Read |
| `allowUnquotedFieldNames` | true, false | Allow unquoted JSON field names. Default is false. | Read |


# CSV

## Reading a CSV file into a DataFrame

In [ ]:
file = "flights/data/summary-data/csv/"
schema = "DEST_COUNTRY_NAME STRING, ORIGIN_COUNTRY_NAME STRING, count INT"
df = (spark.read.format("csv")
.option("header", "true")
.schema(schema)
.option("mode", "FAILFAST") # Exit if any errors
.option("nullValue", "")
 # Replace any null data field with quotes
.load(file))


In [ ]:
df.write.format("csv").mode("overwrite").save("tmp/data/csv/df_csv")

The above generates a folder at the specified location, populated with a bunch of com‐
pressed and compact files.

Similar to JSON and Parquet, we can load a CSV file into a Spark SQL table. 


## CSV data source options

| Property name | Values | Meaning | Scope |
| :--- | :--- | :--- | :--- |
| compression | none, bzip2, deflate, gzip, lz4, or snappy | Use this compression codec for writing. | Write |
| dateFormat | yyyy-MM-dd or DateTimeFormatter | Use this format or any format from Java's DateTimeFormatter. | Read/write |
| multiline | true, false | Use multiline mode. Default is false (single-line mode). | Read |
| inferSchema | true, false | If true, Spark will determine the column data types. Default is false. | Read |
| sep | Any character | Use this character to separate column values in a row. Default delimiter is a comma (,). | Read/write |
| escape | Any character | Use this character to escape quotes. Default is \. | Read/write |
| header | true, false | Indicates whether the first line is a header denoting each column name. Default is false. | Read/write |


# Avro

A built-in data source, the Avro format is used, for exam‐
ple, by Apache Kafka for message serializing and deserializing. It offers many bene‐
fits, including direct mapping to JSON, speed and efficiency, and bindings available
for many programming languages.


In [ ]:
df = (spark.read.format("avro").load("flights/data/summary-data/avro/"))
df.show(truncate=False)


Reading into a Spark SQL table is similar to others. The following shows writing it out.

In [ ]:
(df.write
.format("avro")
.mode("overwrite")
.save("tmp/data/avro/df_avro"))


## Avro options
| Property name | Default value | Meaning | Scope |
| :--- | :--- | :--- | :--- |
| avroSchema | None | Optional Avro schema provided by a user in JSON format. The data type and naming of record fields should match the input Avro data or Catalyst data (Spark internal data type), otherwise the read/write action will fail. | Read/write |
| recordName | topLevelRecord | Top-level record name in write result, which is required in the Avro spec. | Write |
| recordNamespace | "" | Record namespace in write result. | Write |
| ignoreExtension | true | If this option is enabled, all files (with and without the .avro extension) are loaded. Otherwise, files without the .avro extension are ignored. | Read |
| compression | snappy | Allows you to specify the compression codec to use in writing. Currently supported codecs are uncompressed, snappy, deflate, bzip2, and xz. If this option is not set, the value in spark.sql.avro.compression.codec is taken into account. | Write |


# ORC

ORC (Optimized Row Columnar) is a highly efficient, columnar data storage format originally designed for Apache Hive and heavily integrated into Apache Spark.
Similar to Parquet, ORC is specifically optimized for large-scale analytical (OLAP) workloads where queries only need to read a handful of columns out of hundreds, rather than reading entire rows of records. It's usage is similar to the others and we will skip it here.

# Images

Image files are useful for deep learning and machine learning frameworks such as TensorFlow and PyTorch. For computer vision–based machine learning applications, loading and processing image data sets is important.


In [ ]:
from pyspark.ml import image
image_dir = "cctvVideos/train_images/"
images_df = spark.read.format("image").load(image_dir)

In [ ]:
images_df.printSchema()

In [ ]:
images_df.select("image.height", "image.width", "image.nChannels", "image.mode",
"label").show(5, truncate=False)


# Binary files
Spark supports binary files as a data source. The DataFrameReader converts each binary file into a single DataFrame row (record) that contains the raw content and metadata of the file. The binary file data source produces a DataFrame with the following columns:

 * path: StringType
 * modificationTime: TimestampType
 * length: LongType
 * content: BinaryType


## Reading binary files

In [ ]:
path = "cctvVideos/train_images/"
binary_files_df = (spark.read.format("binaryFile")
                   .option("pathGlobFilter", "*.jpg")
                   .load(path))
binary_files_df.show(5)

Note that the label column corresponds to the subfolder name. To ignore partitioning data discovery in a directory, we can set recursiveFile Lookup to "true":


In [ ]:
# In Python
binary_files_df = (spark.read.format("binaryFile")
.option("pathGlobFilter", "*.jpg")
.option("recursiveFileLookup", "true")
.load(path))
binary_files_df.show(5)

In [ ]:
spark.sql("DROP TABLE IF EXISTS us_delay_flights_tbl")

In [ ]:
spark.stop()